In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from openai import OpenAI
from pathlib import Path
import os
from dotenv import load_dotenv
load_dotenv(override=True)

c:\Users\HP\miniconda3\envs\doc-assistant-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\HP\AppData\Local\Temp\ipykernel_16812\4383076.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


True

In [2]:
os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")

In [3]:
def get_text_from_pdf_text(pdf_path:Path) -> str:
    return pdf_path.read_text(encoding='utf-8')

demo_pdf_path = Path(r"..\data\processed\2606.20527v1.txt")
text = get_text_from_pdf_text(demo_pdf_path)
print(text)

StylisticBias: A Few Human Visual Cues Drive Most Social Biases in
MLLMs
Shaghayegh Kolli1,2*
Timo Cavelius1*
Nafiseh Nikeghbal1,2
Samantha Dalal3
Jana Diesner1,2
1Technical University of Munich
2Munich Center for Machine Learning
3Princeton Center for Information and Technology Policy
shaghayegh.kolli@tum.de
Abstract
Multimodal large language models (MLLMs)
are increasingly deployed in personally and
societally consequential settings,
yet the
visual cues that shape how these models
judge people remain poorly understood. Prior
work often compares different (groups of)
individuals, making it difficult to separate
appearance effects from identity differences.
We introduce StylisticBias,
a controlled
benchmark
for
evaluating
attribute-level
social bias in MLLMs.
We generate 500
photorealistic base faces and create about 50
single-attribute variations per face, producing
about 25K images. This design keeps identity
fixed and changes one visual attribute at a
time.
It lets us measure how sp

In [4]:
# import pprint
# embeddings = OpenAIEmbeddings(
#     model="text-embedding-3-small",
#     api_key = os.getenv("OPENAI_API_KEY"),
# )
# try:
#     vector = embeddings.embed_query("Hello World")
# except Exception as e:
#     pprint.pprint(e.__str__())

In [5]:
# embedding_model = GoogleGenerativeAIEmbeddings(api_key = os.getenv("GOOGLE_API_KEY"),
#                                                model="gemini-embedding-2")
# vector = embedding_model.embed_query("hello, world!")
# len(vector)

In [6]:
embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# Can use Qwen3-VL-Embedding-8B but very slow

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3595.86it/s]


In [7]:
def get_recursive_chunks(text:str,chunk_size:int=500,chunk_overlap = 50):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap)
    return splitter.split_text(text)

def get_semantic_chunks(text,embedding_model):
    splitter = SemanticChunker(embeddings=embedding_model)
    return splitter.split_text(text)

In [8]:
recursive_chunks = get_recursive_chunks(text)
semantic_chunks = get_semantic_chunks(text,embedding_model)

In [9]:
print("recursive_chunks",len(recursive_chunks))
print("semantic_chunks",len(semantic_chunks))

recursive_chunks 173
semantic_chunks 36


In [10]:
print("Looking at Recursive character splitted text")
for i,chunk in enumerate(recursive_chunks,1):
    print("="*10)
    print(f"{i}. {chunk}")
    print("="*10)

Looking at Recursive character splitted text
1. StylisticBias: A Few Human Visual Cues Drive Most Social Biases in
MLLMs
Shaghayegh Kolli1,2*
Timo Cavelius1*
Nafiseh Nikeghbal1,2
Samantha Dalal3
Jana Diesner1,2
1Technical University of Munich
2Munich Center for Machine Learning
3Princeton Center for Information and Technology Policy
shaghayegh.kolli@tum.de
Abstract
Multimodal large language models (MLLMs)
are increasingly deployed in personally and
societally consequential settings,
yet the
visual cues that shape how these models
2. yet the
visual cues that shape how these models
judge people remain poorly understood. Prior
work often compares different (groups of)
individuals, making it difficult to separate
appearance effects from identity differences.
We introduce StylisticBias,
a controlled
benchmark
for
evaluating
attribute-level
social bias in MLLMs.
We generate 500
photorealistic base faces and create about 50
single-attribute variations per face, producing
about 25K images. Thi

In [11]:
print("Looking at Semantic chunks")
for i,chunk in enumerate(semantic_chunks,1):
    print("="*10)
    print(f"{i}. {chunk}")
    print("="*10)

Looking at Semantic chunks
1. StylisticBias: A Few Human Visual Cues Drive Most Social Biases in
MLLMs
Shaghayegh Kolli1,2*
Timo Cavelius1*
Nafiseh Nikeghbal1,2
Samantha Dalal3
Jana Diesner1,2
1Technical University of Munich
2Munich Center for Machine Learning
3Princeton Center for Information and Technology Policy
shaghayegh.kolli@tum.de
Abstract
Multimodal large language models (MLLMs)
are increasingly deployed in personally and
societally consequential settings,
yet the
visual cues that shape how these models
judge people remain poorly understood. Prior
work often compares different (groups of)
individuals, making it difficult to separate
appearance effects from identity differences. We introduce StylisticBias,
a controlled
benchmark
for
evaluating
attribute-level
social bias in MLLMs. We generate 500
photorealistic base faces and create about 50
single-attribute variations per face, producing
about 25K images. This design keeps identity
fixed and changes one visual attribute at a
t

In [12]:
import numpy as np

def create_vector_embeddings_given_text(text:str):
    return embedding_model.embed_query(text)

def create_vector_embeddings_given_documents(texts:str):
    return embedding_model.embed_documents(texts)

def get_cosine_similarity(chunk_vector:list[float],query_vector:list[float]) -> float:
    chunk_vector,query_vector = np.asarray(chunk_vector),np.asarray(query_vector)
    return np.dot(chunk_vector,query_vector)*100/(np.linalg.norm(chunk_vector)*np.linalg.norm(query_vector))

In [13]:
recursive_chunks_embeddings = create_vector_embeddings_given_documents(recursive_chunks)
semantic_chunks_embeddings = create_vector_embeddings_given_documents(semantic_chunks)

In [14]:
print("recursive_chunks_embeddings",len(recursive_chunks_embeddings))
print("semantic_chunks_embeddings",len(semantic_chunks_embeddings))

recursive_chunks_embeddings 173
semantic_chunks_embeddings 36


In [15]:
query = "What percentage of reviewed images passed human validation?"
test_embedding = create_vector_embeddings_given_text(query)

In [16]:
# Writing chunks and similarities in text files for comparison

recursive_similarity_scores = []
with open(r'comparison_report/similarity_score_recursive.txt','w',encoding='utf-8') as file:
    for i,emb in enumerate(recursive_chunks_embeddings):
        score = get_cosine_similarity(emb,test_embedding)
        recursive_similarity_scores.append(score)
        
        file.write(f"Comparing chunk {i+1}"+ "\n")
        file.write("="*10 + f" Chunking Score: {score}" + "="*10 + "\n")
        file.write(recursive_chunks[i] + "\n\n")

In [17]:
semantic_similarity_scores = []
with open(r'comparison_report/similarity_score_semantic.txt','w',encoding='utf-8') as file:
    for i,emb in enumerate(semantic_chunks_embeddings):
        score = get_cosine_similarity(emb,test_embedding)
        semantic_similarity_scores.append(score)
        
        file.write(f"Comparing chunk {i+1}"+ "\n")
        file.write("="*10 + f" Chunking Score: {score}" + "="*10 + "\n")
        file.write(semantic_chunks[i] + "\n\n")

In [18]:
max(semantic_similarity_scores),max(recursive_similarity_scores)

(np.float64(38.48772656940333), np.float64(70.35187559245387))